# Finance MiniGPT: A Transformer Built From Scratch on FOMC Statements

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/the-ai-engineer/blob/week03-capstone-v1/capstones/week03_transformers/week03_master_capstone.ipynb)

## 1. Objective and responsible-use boundary

We build a character-level decoder-only Transformer from tensor operations, then inspect one frozen run on official FOMC policy-decision statements. The lesson is about causal attention, optimization, chronological validation, checkpointing, and honest interpretation—not forecasting.

This model is not a Federal Reserve simulator, policy oracle, trading signal, backtest, or source of financial advice. Its generated text is synthetic and often malformed; never present it as authentic Federal Reserve communication.


## 2. Runtime setup and execution mode

Run all cells from top to bottom. In Colab, the bootstrap reuses a valid checkout or clones `week03-capstone-v1`; locally, it locates the existing repository. Dependencies are installed only when missing.

Two visible controls govern ordinary execution: `RUN_MODE = "validation-only"` and `AUTO_DISCONNECT = False`. Validation-only reads frozen inputs and artifacts; it never starts training. CUDA inspection is conditional, so CPU and Apple Silicon remain supported.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

WEEK03_RELEASE_REVISION = os.environ.get("WEEK03_REVISION", "week03-capstone-v1")
WEEK03_REPOSITORY_URL = "https://github.com/FranQuant/the-ai-engineer.git"
WEEK03_REQUIRED_FILES = (
    "capstones/week03_transformers/fomc_statements_2015_2025.txt",
    "capstones/week03_transformers/fomc_statements_2015_2025_manifest.json",
    "capstones/week03_transformers/results/canonical/finance_minigpt_best.pt",
    "capstones/week03_transformers/results/canonical/finance_minigpt_metrics.csv",
    "capstones/week03_transformers/results/canonical/finance_minigpt_samples.json",
    "capstones/week03_transformers/results/canonical/finance_minigpt_training.png",
    "capstones/week03_transformers/results/canonical/finance_minigpt_attention.png",
    "capstones/week03_transformers/results/canonical/finance_minigpt_run.json",
)

def week03_valid_checkout(candidate):
    candidate = Path(candidate).expanduser().resolve()
    return candidate if (candidate / ".git").exists() and all((candidate / item).is_file() for item in WEEK03_REQUIRED_FILES) else None

def week03_find_checkout():
    candidates = []
    configured = os.environ.get("WEEK03_REPO_ROOT")
    if configured:
        candidates.append(Path(configured))
    current = Path.cwd().resolve()
    candidates.extend((current, *current.parents))
    if importlib.util.find_spec("google.colab") is not None:
        candidates.append(Path("/content/the-ai-engineer"))
    for candidate in candidates:
        valid = week03_valid_checkout(candidate)
        if valid is not None:
            return valid
    return None

IN_GOOGLE_COLAB = importlib.util.find_spec("google.colab") is not None
repo_root = week03_find_checkout()
if repo_root is None and IN_GOOGLE_COLAB:
    destination = Path("/content/the-ai-engineer")
    if destination.exists():
        raise RuntimeError(f"Cannot clone Week 3 release: {destination} exists but is not a valid capstone checkout.")
    try:
        subprocess.run(["git", "clone", "--branch", WEEK03_RELEASE_REVISION, "--depth", "1", WEEK03_REPOSITORY_URL, str(destination)], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"Could not clone revision {WEEK03_RELEASE_REVISION!r} from {WEEK03_REPOSITORY_URL}.") from exc
    repo_root = week03_valid_checkout(destination)

if repo_root is None:
    raise RuntimeError("Week 3 repository checkout, frozen corpus, or canonical artifacts were not found. In Colab, rerun this cell after the public release tag is available.")

for package, import_name in (("numpy", "numpy"), ("torch", "torch"), ("matplotlib", "matplotlib")):
    if importlib.util.find_spec(import_name) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", package], check=True)

os.environ["WEEK03_REPO_ROOT"] = str(repo_root)
os.environ.setdefault("WEEK03_RUN_MODE", "validation-only")
os.chdir(repo_root)
print({"colab": IN_GOOGLE_COLAB, "revision": WEEK03_RELEASE_REVISION, "repo_root": str(repo_root), "run_mode": os.environ["WEEK03_RUN_MODE"]})


In [ ]:
from __future__ import annotations

import csv, hashlib, json, math, os, platform, random, re, subprocess, tempfile, time, uuid
from collections import Counter
from dataclasses import asdict, dataclass
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Iterator, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

RUN_MODE = "validation-only"
AUTO_DISCONNECT = False
_requested_run_mode = os.environ.get("WEEK03_RUN_MODE", RUN_MODE)
if _requested_run_mode not in {"validation-only", "phase3b-smoke", "phase4-canonical"}:
    raise ValueError("WEEK03_RUN_MODE must be validation-only, phase3b-smoke, or phase4-canonical.")
RUN_MODE = _requested_run_mode

CONFIG = {
    "phase": "phase4-canonical-interface",
    "run_mode": RUN_MODE,
    "allow_canonical_training": False,
    "seed": 0,
    "architecture": {"block_size": 128, "d_model": 256, "num_heads": 8, "num_layers": 4, "d_ff": 1024, "dropout": 0.1},
    "optimizer_candidates": {"name": "AdamW", "learning_rates": [3e-4, 5e-4], "weight_decay": 0.1, "betas": [0.9, 0.95], "grad_clip": 1.0},
    "training_candidates": {"batch_sizes": [16, 32], "token_budgets": [2_000_000, 5_000_000], "warmup_fraction": 0.05, "min_lr_fraction": 0.1, "validate_every": 250},
    "evaluation": {"validation_split": "2025", "block_sizes_reported": [64, 128, 256]},
    "generation": {"max_new_tokens": 200, "temperature": 0.8, "top_k": 20},
    "artifacts": {
        "best_checkpoint": "finance_minigpt_best.pt", "final_checkpoint": "finance_minigpt_final.pt",
        "metrics": "finance_minigpt_metrics.csv", "run_record": "finance_minigpt_run.json",
        "samples": "finance_minigpt_samples.json", "training_plot": "finance_minigpt_training.png",
        "attention_plot": "finance_minigpt_attention.png",
    },
    "corpus": {"id": "fomc-statements-2015-2025-v2", "sha256": "c632b6a2e7bcfc0360e9fe18113a2ec1c9edce2d42f83b4ff96f5ce4d74ea125"},
}

TEST_RESULTS = {}
def record_check(name: str, condition, detail: str = "") -> bool:
    passed=bool(condition)
    if not passed: raise AssertionError(f"Phase 3A check failed: {name}." + (f" {detail}" if detail else ""))
    TEST_RESULTS[name]=passed; return passed

def select_device() -> torch.device:
    requested = os.environ.get("WEEK03_VALIDATION_DEVICE")
    if requested is not None:
        if RUN_MODE != "validation-only" or requested != "cpu":
            raise ValueError("WEEK03_VALIDATION_DEVICE may only be 'cpu' in validation-only mode.")
        return torch.device("cpu")
    if torch.cuda.is_available(): return torch.device("cuda")
    if hasattr(torch.backends,"mps") and torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True,warn_only=True)
    if hasattr(torch.backends,"cudnn"):
        torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

DEVICE = select_device()
seed_everything(CONFIG["seed"])
runtime_info = {
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "detected_device": str(DEVICE),
    "cuda_device_name": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "in_colab": IN_GOOGLE_COLAB,
    "repository_root": str(repo_root),
    "run_mode": RUN_MODE,
    "auto_disconnect": AUTO_DISCONNECT,
}
print(json.dumps(runtime_info, indent=2))


## 3. Frozen FOMC corpus and chronological split

The committed corpus contains 90 normalized FOMC statements with source and integrity metadata in the manifest. The loader verifies bytes, hashes, document headers, bodies, ordering, and split labels before returning model text.

Statements from 2015–2024 form training; eight 2025 statements form chronological validation. Windows never cross document boundaries. Because 2025 loss was repeatedly observed for checkpoint selection, this is a validation set—not an untouched test set or financial backtest.


In [ ]:
EXPECTED = {
    "corpus_id": "fomc-statements-2015-2025-v2",
    "sha256": "c632b6a2e7bcfc0360e9fe18113a2ec1c9edce2d42f83b4ff96f5ce4d74ea125",
    "bytes": 265669, "characters": 265669, "documents": 90,
    "train": 82, "validation": 8, "extraction_method": "federalreserve_article_paragraphs_v2",
    "normalization_version": "fomc-finance-preserving-v1",
}
START, END = "<|fomc_statement|>", "<|end_fomc_statement|>"

def resolve_corpus_dir() -> Path:
    starts = [Path.cwd()]
    if os.environ.get("WEEK03_REPO_ROOT"): starts.append(Path(os.environ["WEEK03_REPO_ROOT"]).resolve())
    if "__file__" in globals(): starts.append(Path(__file__).resolve().parent)
    for start in starts:
        for parent in [start, *start.parents]:
            for candidate in (parent, parent / "capstones" / "week03_transformers"):
                if (candidate / "fomc_statements_2015_2025.txt").is_file() and (candidate / "fomc_statements_2015_2025_manifest.json").is_file():
                    return candidate
    raise FileNotFoundError("Frozen FOMC corpus files not found from cwd/notebook parents.")

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def load_verified_corpus(corpus_dir: Path):
    corpus_path = corpus_dir / "fomc_statements_2015_2025.txt"
    manifest_path = corpus_dir / "fomc_statements_2015_2025_manifest.json"
    raw_bytes = corpus_path.read_bytes()
    text = raw_bytes.decode("utf-8")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    checks = {
        "sha256": hashlib.sha256(raw_bytes).hexdigest() == EXPECTED["sha256"] == manifest["corpus_sha256"],
        "bytes": len(raw_bytes) == EXPECTED["bytes"] == manifest["corpus_utf8_bytes"],
        "characters": len(text) == EXPECTED["characters"] == manifest["corpus_characters"],
        "corpus_id": manifest["corpus_id"] == EXPECTED["corpus_id"],
        "extraction_method": manifest["extraction_method"] == EXPECTED["extraction_method"],
        "normalization_version": manifest["normalization_version"] == EXPECTED["normalization_version"],
        "document_count": manifest["document_count"] == EXPECTED["documents"] == len(manifest["documents"]),
        "split_counts": manifest["train_document_count"] == EXPECTED["train"] and manifest["validation_document_count"] == EXPECTED["validation"],
        "zero_failures": len(manifest["failures"]) == 0,
        "marker_counts": text.count(START) == text.count(END) == EXPECTED["documents"],
    }
    failed = [name for name, ok in checks.items() if not ok]
    if failed: raise ValueError(f"Corpus integrity mismatch: {failed}")

    pattern = re.compile(
        rf"{re.escape(START)}\n"
        r"date: (?P<date>\d{4}-\d{2}-\d{2})\n"
        r"meeting_type: (?P<meeting_type>scheduled|unscheduled)\n"
        r"document_id: (?P<document_id>[^\n]+)\n\n"
        rf"(?P<body>.*?)(?=\n{re.escape(END)})\n{re.escape(END)}",
        re.DOTALL,
    )
    matches = list(pattern.finditer(text))
    reconstructed = "\n\n".join(m.group(0) for m in matches) + "\n"
    if len(matches) != EXPECTED["documents"] or reconstructed != text:
        raise ValueError("Structural marker parse was not lossless.")
    records = []
    for match, meta in zip(matches, manifest["documents"]):
        record = {"document_id": match["document_id"], "statement_date": match["date"], "meeting_type": match["meeting_type"], "split": meta["split"], "body": match["body"]}
        expected_fields = (meta["document_id"], meta["statement_date"], meta["meeting_type"])
        if (record["document_id"], record["statement_date"], record["meeting_type"]) != expected_fields:
            raise ValueError(f"Document order/header mismatch at {meta['document_id']}")
        if len(record["body"]) != meta["normalized_characters"] or sha256_text(record["body"]) != meta["normalized_sha256"]:
            raise ValueError(f"Body integrity mismatch for {record['document_id']}")
        expected_split = "train" if record["statement_date"] <= "2024-12-31" else "validation"
        if record["split"] != expected_split or (record["split"] == "validation") != record["statement_date"].startswith("2025-"):
            raise ValueError(f"Split/date mismatch for {record['document_id']}")
        records.append(record)
    if [r["statement_date"] for r in records] != sorted(r["statement_date"] for r in records):
        raise ValueError("Documents are not in chronological order.")
    return records, manifest, checks

CORPUS_DIR = resolve_corpus_dir()
documents, manifest, integrity_checks = load_verified_corpus(CORPUS_DIR)
print("Corpus integrity PASS:", all(integrity_checks.values()), "|", CORPUS_DIR)


In [ ]:
train_docs = [d for d in documents if d["split"] == "train"]
val_docs = [d for d in documents if d["split"] == "validation"]
assert {d["document_id"] for d in train_docs}.isdisjoint(d["document_id"] for d in val_docs)
assert len(train_docs) == 82 and len(val_docs) == 8
assert sum(len(d["body"]) for d in train_docs) == 237840
assert sum(len(d["body"]) for d in val_docs) == 17474
assert Counter(d["meeting_type"] for d in documents) == {"scheduled": 87, "unscheduled": 3}

def available_windows(records, block_size): return sum(max(0, len(d["body"]) - block_size) for d in records)
summary_rows = []
for name, records in (("train", train_docs), ("validation", val_docs)):
    summary_rows.append({"split": name, "documents": len(records), "date_range": f"{records[0]['statement_date']} to {records[-1]['statement_date']}", "body_characters": sum(len(d['body']) for d in records), "windows@128": available_windows(records, 128)})
for row in summary_rows: print(row)


## 4. Character tokenizer and vocabulary

The vocabulary is the sorted set of characters found only in training statement bodies. `encode` maps characters to integer IDs and `decode` reverses that mapping. Structural markers and metadata are excluded, while validation text and the five fixed prompts must be covered by the training vocabulary.


In [ ]:
EVAL_PROMPTS = [
    "The Committee decided to", "Inflation has", "The labor market",
    "The target range for the federal funds rate",
    "In assessing the appropriate stance of monetary policy",
]
training_bodies = tuple(d["body"] for d in train_docs)
training_body_chars = sorted(set().union(*(set(body) for body in training_bodies)))
chars = training_body_chars.copy()
stoi = {ch: i for i, ch in enumerate(chars)}; itos = {i: ch for ch, i in stoi.items()}
VOCAB_SIZE = len(chars)

def encode(text: str) -> list[int]:
    unknown = sorted(set(text) - set(stoi))
    if unknown: raise ValueError(f"Unknown characters: {unknown!r}")
    return [stoi[ch] for ch in text]

def decode(ids: Sequence[int]) -> str:
    try: return "".join(itos[int(i)] for i in ids)
    except KeyError as exc: raise ValueError(f"Token ID outside vocabulary: {exc.args[0]}") from exc

serialized_headers = tuple(
    line
    for d in documents
    for line in (
        START, f'date: {d["statement_date"]}', f'meeting_type: {d["meeting_type"]}',
        f'document_id: {d["document_id"]}', END,
    )
)
model_documents = training_bodies
tokenizer_inputs = training_bodies
marker_exclusion = (
    chars == sorted(set().union(*(set(body) for body in training_bodies)))
    and model_documents == tuple(d["body"] for d in train_docs)
    and tokenizer_inputs == tuple(d["body"] for d in train_docs)
    and all(START not in body and END not in body for body in model_documents)
    and all(header not in body.splitlines() for body in model_documents for header in serialized_headers)
    and all(not body.startswith((START, "date: ", "meeting_type: ", "document_id: ")) for body in model_documents)
)
record_check("body-only vocabulary exactness", chars == training_body_chars)
record_check("structural marker/header exclusion", marker_exclusion)
record_check("training vocabulary size", VOCAB_SIZE == 73)
record_check("tokenizer train round-trip", all(decode(encode(d["body"])) == d["body"] for d in train_docs))
record_check("validation character coverage", not (set().union(*(set(d["body"]) for d in val_docs)) - set(chars)))
record_check("prompt character coverage", all(decode(encode(prompt)) == prompt for prompt in EVAL_PROMPTS))
VOCABULARY_DATA = {"chars": chars, "stoi": stoi}
print({"vocabulary": VOCAB_SIZE, "body_only_exact": "PASS", "marker_header_exclusion": "PASS", "train_round_trip": "PASS", "validation_coverage": "PASS", "prompt_coverage": "PASS"})


## 5. Scaled causal attention

For `Q`, `K`, and `V` shaped `[batch, heads, time, head_dim]`, attention forms `[batch, heads, time, time]` scores, scales by `sqrt(head_dim)`, masks future keys, applies softmax, and mixes values. `True` means an allowed key.

The triangular mask is what makes next-character prediction causal: position `t` may use positions `≤ t`, never future positions. A two-token NumPy reference makes the masking and probabilities hand-checkable.


In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None, dropout_p=0.0, training=False, return_weights=False):
    if q.ndim != 4 or k.ndim != 4 or v.ndim != 4: raise ValueError("Q, K, V must each have shape [B,H,T,d_head].")
    if q.shape != k.shape or q.shape != v.shape: raise ValueError(f"Q, K, V shapes must match; got {q.shape}, {k.shape}, {v.shape}.")
    if not q.is_floating_point() or q.dtype != k.dtype or q.dtype != v.dtype: raise TypeError("Q, K, V must share a floating dtype.")
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))  # [B,H,T,T]
    if mask is not None:
        if mask.dtype != torch.bool: raise TypeError("Attention mask must be boolean (True=allowed).")
        try: allowed = torch.broadcast_to(mask.to(scores.device), scores.shape)
        except RuntimeError as exc: raise ValueError(f"Mask shape {tuple(mask.shape)} is not broadcastable to scores {tuple(scores.shape)}.") from exc
        if not allowed.any(dim=-1).all(): raise ValueError("Every attention row must allow at least one key.")
        scores = scores.masked_fill(~allowed, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    mixed_weights = F.dropout(weights, p=dropout_p, training=training)
    output = mixed_weights @ v  # [B,H,T,d_head]
    return (output, weights) if return_weights else output

q_small = torch.tensor([[[[1., 0.], [0., 1.]]]])
k_small = torch.tensor([[[[1., 0.], [0., 1.]]]])
v_small = torch.tensor([[[[1., 2.], [3., 4.]]]])
scores_small = q_small @ k_small.transpose(-2, -1) / math.sqrt(2)
_, unmasked = scaled_dot_product_attention(q_small, k_small, v_small, return_weights=True)
causal2 = torch.tril(torch.ones(2, 2, dtype=torch.bool))[None, None]
out_masked, masked = scaled_dot_product_attention(q_small, k_small, v_small, mask=causal2, return_weights=True)

np_scores = np.array([[1., 0.], [0., 1.]]) / np.sqrt(2.0)
np_shift = np_scores - np_scores.max(axis=-1, keepdims=True)
np_unmasked = np.exp(np_shift) / np.exp(np_shift).sum(axis=-1, keepdims=True)
expected_unmasked = np.array([[0.66976155, 0.33023845], [0.33023845, 0.66976155]])
expected_masked = np.array([[1.0, 0.0], [0.33023845, 0.66976155]])
expected_output = expected_masked @ np.array([[1., 2.], [3., 4.]])
record_check("numerical attention shapes", scores_small.shape == (1,1,2,2) and out_masked.shape == (1,1,2,2))
record_check("independent numerical attention reference", np.allclose(np_unmasked, expected_unmasked, atol=1e-7) and np.allclose(unmasked.numpy()[0,0], expected_unmasked, atol=1e-7) and np.allclose(out_masked.numpy()[0,0], expected_output, atol=1e-7))
record_check("attention rows normalize", torch.allclose(masked.sum(-1), torch.ones_like(masked.sum(-1))))
record_check("future attention is zero", np.allclose(masked.numpy()[0,0], expected_masked, atol=1e-7) and torch.count_nonzero(masked[..., 0, 1]) == 0)
print("Q=", q_small, "\nK=", k_small, "\nV=", v_small)
print("scaled scores=", scores_small, "\nunmasked=", unmasked, "\ncausal=", masked)
print("Numerical attention PASS")


## 6. Multi-head attention

A single linear projection creates `Q/K/V`, reshaped from `[B,T,3D]` into `[3,B,H,T,d_head]`. Each head applies the causal attention function independently; concatenated head outputs return to `[B,T,D]` before the output projection. Multiple heads let the block allocate separate representational subspaces without relaxing the causal mask.


In [ ]:
class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout, max_seq_len):
        super().__init__()
        if d_model % num_heads: raise ValueError("d_model must be divisible by num_heads.")
        self.num_heads, self.head_dim = num_heads, d_model // num_heads
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout, self.resid_dropout = dropout, nn.Dropout(dropout)
        self.register_buffer("causal_mask", torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))[None,None])
    def forward(self, x, return_attention=False):
        if x.ndim != 3: raise ValueError("Attention input must have shape [B,T,D].")
        B,T,D = x.shape
        if T > self.causal_mask.size(-1): raise ValueError(f"Context length {T} exceeds configured maximum {self.causal_mask.size(-1)}.")
        qkv = self.qkv(x).view(B,T,3,self.num_heads,self.head_dim).permute(2,0,3,1,4)
        q,k,v = qkv.unbind(0)  # each [B,H,T,d_head]
        y, weights = scaled_dot_product_attention(q,k,v,self.causal_mask[:,:,:T,:T],self.attn_dropout,self.training,True)
        y = y.transpose(1,2).contiguous().view(B,T,D)
        y = self.resid_dropout(self.out_proj(y))
        return (y, weights) if return_attention else y


## 7. Transformer block and Finance MiniGPT

Sinusoidal positions supply token order. Each Pre-LN block applies `x + attention(LN(x))`, then `x + FFN(LN(x))`; the feed-forward path expands `D → d_ff → D` with GELU. Finance MiniGPT stacks these blocks over token embeddings, applies a final LayerNorm, and ties the output head to the token embedding.

Input IDs have shape `[B,T]`; logits have shape `[B,T,V]`. The bounded checks below verify shapes, parameter sharing, initialization, gradients, causal non-leakage, and one tiny optimizer update. That tiny update tests implementation mechanics only—it is not canonical or smoke training.


In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len):
        super().__init__()
        if d_model <= 0 or max_seq_len <= 0: raise ValueError("d_model and max_seq_len must be positive.")
        position = torch.arange(max_seq_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0,d_model,2,dtype=torch.float32) * (-math.log(10000.0)/d_model))
        pe = torch.zeros(max_seq_len,d_model); pe[:,0::2] = torch.sin(position*div)
        pe[:,1::2] = torch.cos(position*div[:pe[:,1::2].shape[1]])
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        if x.ndim != 3: raise ValueError("Positional input must have shape [B,T,D].")
        if x.size(1) > self.pe.size(1): raise ValueError(f"Context length {x.size(1)} exceeds positional maximum {self.pe.size(1)}.")
        if x.size(2) != self.pe.size(2): raise ValueError("Embedding dimension does not match positional encoding.")
        return x + self.pe[:,:x.size(1)].to(dtype=x.dtype)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout, max_seq_len):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = MultiHeadCausalSelfAttention(d_model,num_heads,dropout,max_seq_len)
        self.ffn = nn.Sequential(nn.Linear(d_model,d_ff), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_ff,d_model), nn.Dropout(dropout))
    def forward(self, x, return_attention=False):
        attn_result = self.attn(self.ln1(x), return_attention=return_attention)
        if return_attention: attn_out, weights = attn_result
        else: attn_out = attn_result
        x = x + attn_out; x = x + self.ffn(self.ln2(x))
        return (x, weights) if return_attention else x


In [ ]:
class FinanceMiniGPT(nn.Module):
    def __init__(self, vocab_size, architecture):
        super().__init__(); self.vocab_size = vocab_size; self.architecture = dict(architecture)
        D, L, H, FF, BS, P = (architecture[k] for k in ("d_model","num_layers","num_heads","d_ff","block_size","dropout"))
        self.token_embedding = nn.Embedding(vocab_size,D)
        self.position = SinusoidalPositionalEncoding(D,BS); self.dropout = nn.Dropout(P)
        self.blocks = nn.ModuleList([TransformerBlock(D,H,FF,P,BS) for _ in range(L)])
        self.ln_f = nn.LayerNorm(D); self.lm_head = nn.Linear(D,vocab_size,bias=False)
        self.apply(self._init_weights); self.lm_head.weight = self.token_embedding.weight
        self.architecture["parameter_count"] = sum(p.numel() for p in self.parameters())
    @staticmethod
    def _init_weights(module):
        if isinstance(module,(nn.Linear,nn.Embedding)): nn.init.normal_(module.weight,0.0,0.02)
        if isinstance(module,nn.Linear) and module.bias is not None: nn.init.zeros_(module.bias)
        if isinstance(module,nn.LayerNorm): nn.init.ones_(module.weight); nn.init.zeros_(module.bias)
    def forward(self, idx, targets=None, return_attentions=False):
        if idx.ndim != 2: raise ValueError("Token input must have rank 2: [B,T].")
        if idx.dtype not in (torch.int8,torch.int16,torch.int32,torch.int64,torch.uint8): raise TypeError("Token input must use an integer dtype.")
        if idx.numel() and (idx.min().item() < 0 or idx.max().item() >= self.vocab_size): raise ValueError(f"Token IDs must be in [0,{self.vocab_size-1}].")
        if idx.size(1) > self.architecture["block_size"]: raise ValueError(f"Context length {idx.size(1)} exceeds block_size={self.architecture['block_size']}; crop only in generation.")
        if targets is not None and (targets.shape != idx.shape or targets.dtype != torch.long): raise ValueError("Targets must be torch.long with the same [B,T] shape as input.")
        x = self.dropout(self.position(self.token_embedding(idx)))
        maps=[]
        for block in self.blocks:
            if return_attentions: x,w = block(x,True); maps.append(w)
            else: x = block(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None if targets is None else F.cross_entropy(logits.reshape(-1,self.vocab_size),targets.reshape(-1))
        return {"logits": logits, "loss": loss, "attentions": maps if return_attentions else None}


In [ ]:
TINY_ARCH = {"block_size": 16,"d_model": 32,"num_heads": 4,"num_layers": 2,"d_ff": 64,"dropout": 0.0}
seed_everything(0); tiny_model = FinanceMiniGPT(VOCAB_SIZE,TINY_ARCH).to(DEVICE)
sample = torch.tensor([encode(train_docs[0]["body"][:12])],dtype=torch.long,device=DEVICE)
targets = torch.tensor([encode(train_docs[0]["body"][1:13])],dtype=torch.long,device=DEVICE)
mha = MultiHeadCausalSelfAttention(32,4,0.0,16).to(DEVICE)
record_check("multi-head output shape", mha(torch.randn(2,7,32,device=DEVICE)).shape == (2,7,32))
block = TransformerBlock(32,4,64,0.0,16).to(DEVICE)
record_check("block shape invariance", block(torch.randn(2,7,32,device=DEVICE)).shape == (2,7,32))
result=tiny_model(sample,targets)
record_check("logits shape", result["logits"].shape==(1,12,VOCAB_SIZE))
record_check("finite scalar loss", result["loss"].ndim==0 and torch.isfinite(result["loss"]))
tiny_model.zero_grad(set_to_none=True); result["loss"].backward()
grads=[p.grad for p in tiny_model.parameters() if p.grad is not None]
record_check("finite nonzero gradients", grads and all(torch.isfinite(g).all() for g in grads) and any(torch.count_nonzero(g)>0 for g in grads))
optimizer=torch.optim.AdamW(tiny_model.parameters(),lr=1e-3); before=[p.detach().clone() for p in tiny_model.parameters()]
optimizer.step()
record_check("optimizer step changes parameters", any(not torch.equal(a,b) for a,b in zip(before,tiny_model.parameters())))
tiny_model.eval(); a=sample.clone(); b=sample.clone(); b[0,-1]=(b[0,-1]+1)%VOCAB_SIZE
with torch.no_grad(): la=tiny_model(a)["logits"]; lb=tiny_model(b)["logits"]
record_check("causal non-leakage", torch.equal(la[:,:-1],lb[:,:-1]))
overflow_raised = False
try: tiny_model(torch.zeros(1,17,dtype=torch.long,device=DEVICE))
except ValueError as exc: overflow_raised = "block_size" in str(exc)
record_check("context overflow rejection", overflow_raised)
record_check("weight tying storage", tiny_model.lm_head.weight.data_ptr()==tiny_model.token_embedding.weight.data_ptr())
record_check("serialized architecture parameter count", tiny_model.architecture["parameter_count"] == sum(p.numel() for p in tiny_model.parameters()))

seed_everything(7)
init_model = FinanceMiniGPT(VOCAB_SIZE,TINY_ARCH)
initialized_weights = [m.weight.detach().flatten() for m in init_model.modules() if isinstance(m, (nn.Linear, nn.Embedding))]
all_initialized = torch.cat(initialized_weights)
init_statistics_ok = abs(float(all_initialized.mean())) < 0.01 and 0.012 < float(all_initialized.std()) < 0.028
zero_linear_biases = all(torch.count_nonzero(m.bias) == 0 for m in init_model.modules() if isinstance(m, nn.Linear) and m.bias is not None)
record_check("GPT-style initialization statistics", init_statistics_ok, f"mean={float(all_initialized.mean()):.5f}, std={float(all_initialized.std()):.5f}")
record_check("zero linear biases", zero_linear_biases)

dropout_arch = {**TINY_ARCH, "dropout": 0.5}
seed_everything(11); dropout_model = FinanceMiniGPT(VOCAB_SIZE,dropout_arch).to(DEVICE)
dropout_model.train()
with torch.no_grad(): train_a=dropout_model(sample)["logits"]; train_b=dropout_model(sample)["logits"]
dropout_model.eval()
with torch.no_grad(): eval_a=dropout_model(sample)["logits"]; eval_b=dropout_model(sample)["logits"]
record_check("dropout train stochasticity", not torch.equal(train_a,train_b))
record_check("dropout eval determinism", torch.equal(eval_a,eval_b))
print("Architecture, initialization, gradient, and dropout tests PASS")


## 8. Document-aware training batches

Each dataset item is a shifted `(x, y)` pair of `block_size` characters from one statement body. The index stores `(document, start)` locations rather than duplicating tensors. Random training batches use a supplied generator; exhaustive validation walks the complete deterministic index.


In [ ]:
class DocumentWindowDataset(Dataset):
    def __init__(self, records, block_size, stoi):
        if block_size < 1: raise ValueError("block_size must be positive.")
        self.block_size=block_size; self.encoded=[]; self.index=[]
        for doc_i,d in enumerate(records):
            ids=torch.tensor([stoi[ch] for ch in d["body"]],dtype=torch.long); self.encoded.append(ids)
            self.index.extend((doc_i,start) for start in range(max(0,len(ids)-block_size)))
        if not self.index: raise ValueError(f"No document has at least block_size+1={block_size+1} characters.")
    def __len__(self): return len(self.index)
    def __getitem__(self,i):
        doc_i,start=self.index[i]; ids=self.encoded[doc_i]; return ids[start:start+self.block_size],ids[start+1:start+self.block_size+1]
    def sample_batch(self,batch_size,generator,device):
        choices=torch.randint(len(self),(batch_size,),generator=generator)
        pairs=[self[int(i)] for i in choices]; return torch.stack([p[0] for p in pairs]).to(device),torch.stack([p[1] for p in pairs]).to(device)

for bs in (64,128,256):
    train_n=available_windows(train_docs,bs); val_n=available_windows(val_docs,bs)
    assert train_n==len(DocumentWindowDataset(train_docs,bs,stoi)) and val_n==len(DocumentWindowDataset(val_docs,bs,stoi))
    print({"block_size":bs,"train_windows":train_n,"validation_windows":val_n})


## 9. Frozen canonical configuration

The table separates this notebook’s current validation device from the hardware and settings recorded for the immutable canonical run. Hardware detection never rewrites the frozen architecture or hyperparameters.

- **CPU:** supported for canonical validation.
- **MPS:** compatible with local validation.
- **CUDA:** compatible with experimental training code, but this release notebook performs no training by default.


In [ ]:
CANONICAL_ARCH={"block_size":128,"d_model":256,"num_heads":8,"num_layers":4,"d_ff":1024,"dropout":0.1}
CANONICAL_TRAINING={"batch_size":32,"steps":1221,"lr":3e-4,"min_lr":3e-5,"warmup_steps":61,"weight_decay":0.1,"betas":[0.9,0.95],"grad_clip":1.0,"seed":0,"validation_steps":[0,250,500,750,1000,1220],"eval_batch_size":128}
CANONICAL_TOKEN_BUDGET=5_001_216
assert CANONICAL_TRAINING["steps"]*CANONICAL_TRAINING["batch_size"]*CANONICAL_ARCH["block_size"]==CANONICAL_TOKEN_BUDGET


from IPython.display import Image, Markdown, display

def markdown_table(headers, rows):
    head = "| " + " | ".join(headers) + " |"
    rule = "| " + " | ".join("---" for _ in headers) + " |"
    body = ["| " + " | ".join(str(value).replace("|", "\\|") for value in row) + " |" for row in rows]
    return "\n".join([head, rule, *body])

canonical_configuration_rows = [
    ("Current execution", RUN_MODE),
    ("Validation device", str(DEVICE)),
    ("Recorded canonical hardware", "CUDA"),
    ("Recorded precision", "FP32 (AMP disabled)"),
    ("Context / width", f'{CANONICAL_ARCH["block_size"]} / {CANONICAL_ARCH["d_model"]}'),
    ("Layers / heads", f'{CANONICAL_ARCH["num_layers"]} / {CANONICAL_ARCH["num_heads"]}'),
    ("Feed-forward width / dropout", f'{CANONICAL_ARCH["d_ff"]} / {CANONICAL_ARCH["dropout"]}'),
    ("Parameters", "3,178,240"),
    ("Batch size / steps", f'{CANONICAL_TRAINING["batch_size"]} / {CANONICAL_TRAINING["steps"]}'),
    ("AdamW learning rate", CANONICAL_TRAINING["lr"]),
    ("Warmup / minimum LR", f'{CANONICAL_TRAINING["warmup_steps"]} steps / {CANONICAL_TRAINING["min_lr"]}'),
    ("Validation steps", CANONICAL_TRAINING["validation_steps"]),
    ("Token budget", f'{CANONICAL_TOKEN_BUDGET:,}'),
]
display(Markdown(markdown_table(["Context", "Frozen value"], canonical_configuration_rows)))



## 10. Anatomy of one optimization step

The real `train_model` function implements this sequence; the excerpt is explanatory and does not execute an update:

```python
x, y = train_ds.sample_batch(batch_size, generator, device)  # training data only
optimizer.zero_grad(set_to_none=True)                         # clear old gradients
loss = model(x, y)["loss"]                                  # forward + cross-entropy
loss.backward()                                               # reverse-mode autodiff
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
optimizer.step()                                              # AdamW parameter update
lr = cosine_lr(step, steps, warmup_steps, max_lr, min_lr)     # scheduled learning rate
for group in optimizer.param_groups:
    group["lr"] = lr
```

In the executable loop, the scheduled learning rate is assigned at the beginning of each step, before the forward pass. Validation data is evaluated under `torch.no_grad()` and is never passed to `optimizer.step()`.


## 11. Training, validation, and checkpoint protocol

AdamW applies weight decay to matrix weights while excluding biases, LayerNorm parameters, and the tied embedding. The schedule uses linear warmup followed by cosine decay. At steps `0, 250, 500, 750, 1000, 1220`, exhaustive 2025 validation CE is computed; the lowest CE selects the best checkpoint.

The best checkpoint—not merely the last state—is used for reported inference. Atomic temporary-file replacement prevents partial writes, and saved verification logits support reload-equivalence checking. Authorization guards remain active, but validation-only exits before model construction, batch sampling, optimizer creation, or artifact writing for a training run.


In [ ]:
def adamw_parameter_groups(model, weight_decay):
    decay=[]; no_decay=[]; membership={}
    for name,p in model.named_parameters():
        if not p.requires_grad: continue
        use_decay=p.ndim>=2 and name!="token_embedding.weight"
        (decay if use_decay else no_decay).append(p); membership[name]="decay" if use_decay else "no_decay"
    decay_ids,no_decay_ids={id(p) for p in decay},{id(p) for p in no_decay}; trainable_ids={id(p) for p in model.parameters() if p.requires_grad}
    if decay_ids & no_decay_ids: raise AssertionError("AdamW parameter groups overlap.")
    if decay_ids | no_decay_ids != trainable_ids: raise AssertionError("AdamW parameter groups do not cover every trainable parameter exactly once.")
    return [{"params":decay,"weight_decay":weight_decay,"group_name":"decay"},{"params":no_decay,"weight_decay":0.0,"group_name":"no_decay"}],membership

def configure_adamw(model,lr,weight_decay,betas=(0.9,0.95)):
    groups,membership=adamw_parameter_groups(model,weight_decay)
    return torch.optim.AdamW(groups,lr=lr,betas=betas),membership

def cosine_lr(step,total_steps,warmup_steps,max_lr,min_lr):
    """Warmup step 0 uses max_lr/warmup_steps; zero warmup starts at max_lr. The final optimization step is min_lr."""
    if total_steps<1: raise ValueError("total_steps must be positive.")
    if not 0<=step<total_steps: raise ValueError("step must be in [0, total_steps).")
    if not 0<=warmup_steps<total_steps: raise ValueError("warmup_steps must be in [0, total_steps).")
    if not (math.isfinite(max_lr) and math.isfinite(min_lr) and 0<=min_lr<=max_lr): raise ValueError("Learning rates must be finite with 0 <= min_lr <= max_lr.")
    if warmup_steps>0 and step<warmup_steps: return max_lr*(step+1)/warmup_steps
    if total_steps==1: return min_lr
    denominator=total_steps-1-warmup_steps
    if denominator<=0: return min_lr
    progress=(step-warmup_steps)/denominator
    return min_lr+0.5*(max_lr-min_lr)*(1+math.cos(math.pi*progress))


In [ ]:
def safe_perplexity(cross_entropy):
    if not math.isfinite(cross_entropy): raise FloatingPointError("Cross-entropy must be finite before perplexity.")
    return math.exp(cross_entropy) if cross_entropy < math.log(np.finfo(np.float64).max) else float("inf")

@torch.no_grad()
def evaluate_exhaustive_details(model,dataset,batch_size,device):
    prior=model.training; model.eval(); weighted=0.0; tokens=0
    try:
        for start in range(0,len(dataset),batch_size):
            pairs=[dataset[i] for i in range(start,min(start+batch_size,len(dataset)))]
            x=torch.stack([p[0] for p in pairs]).to(device); y=torch.stack([p[1] for p in pairs]).to(device)
            loss=model(x,y)["loss"]
            if not torch.isfinite(loss): raise FloatingPointError("Non-finite exhaustive validation loss.")
            weighted+=float(loss.detach().cpu())*y.numel(); tokens+=y.numel()
        ce=weighted/tokens
        return {"cross_entropy":ce,"perplexity":safe_perplexity(ce),"windows":len(dataset),"tokens":tokens}
    finally: model.train(prior)

def evaluate_exhaustive(model,dataset,batch_size,device): return evaluate_exhaustive_details(model,dataset,batch_size,device)["cross_entropy"]

def enter_training_mode(model,allowed):
    if not allowed: raise RuntimeError("Training requires an authorized resolved run configuration.")
    model.train()

def train_model(model,train_ds,val_ds,training,checkpoint_context,device):
    enter_training_mode(model,training.get("authorized_training") is True and training.get("run_mode") in {"phase3b-smoke","phase4-canonical"})
    opt,_=configure_adamw(model,training["lr"],training["weight_decay"],tuple(training["betas"])); gen=torch.Generator().manual_seed(training["seed"])
    metrics=[]; best_loss=float("inf"); best_step=None; tokens=0; started=time.perf_counter(); validation_steps=set(training["validation_steps"])
    for step in range(training["steps"]):
        lr=cosine_lr(step,training["steps"],training["warmup_steps"],training["lr"],training["min_lr"])
        for group in opt.param_groups: group["lr"]=lr
        x,y=train_ds.sample_batch(training["batch_size"],gen,device); opt.zero_grad(set_to_none=True)
        loss=model(x,y)["loss"]
        if not torch.isfinite(loss): raise FloatingPointError(f"Non-finite training minibatch loss at step {step}")
        loss.backward(); grads=[p.grad for p in model.parameters() if p.grad is not None]
        if not grads or not all(torch.isfinite(g).all() for g in grads): raise FloatingPointError(f"Non-finite/missing gradients at step {step}")
        grad_norm=torch.nn.utils.clip_grad_norm_(model.parameters(),training["grad_clip"])
        if not torch.isfinite(grad_norm): raise FloatingPointError(f"Non-finite gradient norm at step {step}")
        opt.step(); tokens+=y.numel()
        elapsed=time.perf_counter()-started
        row={"step":step,"train_minibatch_ce":float(loss.detach().cpu()),"gradient_norm_before_clipping":float(grad_norm.detach().cpu()),"validation_ce":None,"validation_perplexity":None,"lr":lr,"tokens_processed":tokens,"elapsed_seconds":elapsed,"throughput_tokens_per_second":tokens/elapsed if elapsed>0 else None}
        if step in validation_steps:
            validation=evaluate_exhaustive_details(model,val_ds,training["eval_batch_size"],device)
            row.update({"validation_ce":validation["cross_entropy"],"validation_perplexity":validation["perplexity"]})
            metrics.append(row)
            if validation["cross_entropy"]<best_loss:
                best_loss,best_step=validation["cross_entropy"],step
                save_checkpoint(training["best_path"],"best",model,opt,None,training,checkpoint_context,step,best_step,best_loss,metrics)
            continue
        metrics.append(row)
    if best_step is None: raise RuntimeError("No validation checkpoint was produced.")
    save_checkpoint(training["final_path"],"final",model,opt,None,training,checkpoint_context,training["steps"]-1,best_step,best_loss,metrics)
    return {"metrics":metrics,"best_step":best_step,"best_validation_loss":best_loss,"tokens_processed":tokens,"elapsed_seconds":time.perf_counter()-started}


In [ ]:
def environment_metadata():
    return {"python":platform.python_version(),"torch":torch.__version__,"numpy":np.__version__,"platform":platform.platform(),"device":str(DEVICE),"cuda_available":torch.cuda.is_available(),"mps_available":bool(hasattr(torch.backends,"mps") and torch.backends.mps.is_available())}

def atomic_torch_save(payload,path):
    path=Path(path); temp=path.with_name(path.name+".tmp")
    torch.save(payload,temp); os.replace(temp,path)

def atomic_json_write(payload,path):
    path=Path(path); temp=path.with_name(path.name+".tmp")
    temp.write_text(json.dumps(payload,indent=2,sort_keys=True)+"\n",encoding="utf-8"); os.replace(temp,path)

def file_sha256(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda:handle.read(1<<20),b""): digest.update(chunk)
    return digest.hexdigest()

def artifact_record(path):
    path=Path(path); return {"filename":path.name,"size_bytes":path.stat().st_size,"sha256":file_sha256(path)}

CHECKPOINT_REQUIRED={"artifact_version","checkpoint_kind","run_mode","run_id","model_state","architecture_config","parameter_count","training_config","vocabulary","corpus_id","corpus_sha256","split_metadata","seed","current_step","best_step","best_validation_loss","optimizer_state","scheduler_state","metrics_history","environment","verification_ids","verification_logits"}

def save_checkpoint(path,kind,model,optimizer,scheduler,training_config,context,step,best_step,best_loss,metrics):
    if kind not in {"best","final","phase3a-test"}: raise ValueError("Checkpoint kind must be best, final, or phase3a-test.")
    verification_ids=context.get("verification_ids")
    verification_logits=None
    if verification_ids is not None:
        prior=model.training; model.eval()
        try:
            with torch.no_grad(): verification_logits=model(torch.tensor([verification_ids],dtype=torch.long,device=next(model.parameters()).device))["logits"].detach().cpu()
        finally: model.train(prior)
    payload={"artifact_version":"finance-minigpt-checkpoint-v3","checkpoint_kind":kind,"run_mode":training_config.get("run_mode","validation-only"),"run_id":training_config.get("run_id"),"model_state":model.state_dict(),"architecture_config":model.architecture,"parameter_count":sum(p.numel() for p in model.parameters()),"training_config":training_config,"vocabulary":context["vocabulary"],"corpus_id":context["corpus_id"],"corpus_sha256":context["corpus_sha256"],"split_metadata":context["split_metadata"],"seed":context["seed"],"current_step":step,"best_step":best_step,"best_validation_loss":best_loss,"optimizer_state":optimizer.state_dict() if kind=="final" else None,"scheduler_state":scheduler.state_dict() if kind=="final" and scheduler is not None else None,"metrics_history":metrics,"environment":environment_metadata(),"verification_ids":verification_ids,"verification_logits":verification_logits}
    atomic_torch_save(payload,Path(path))

def load_checkpoint(path,map_location="cpu"):
    payload=torch.load(Path(path),map_location=map_location,weights_only=False)
    if payload.get("artifact_version") not in {"finance-minigpt-checkpoint-v1","finance-minigpt-checkpoint-v2","finance-minigpt-checkpoint-v3"}: raise ValueError("Unsupported checkpoint schema.")
    missing=CHECKPOINT_REQUIRED-payload.keys() if payload.get("artifact_version")=="finance-minigpt-checkpoint-v3" else set()
    if missing: raise ValueError(f"Checkpoint missing required fields: {sorted(missing)}")
    if payload.get("checkpoint_kind") not in {"best","final","phase3a-test"}: raise ValueError("Invalid checkpoint kind.")
    if payload.get("checkpoint_kind")=="best" and (payload.get("optimizer_state") is not None or payload.get("scheduler_state") is not None): raise ValueError("Compact best checkpoint must not contain optimizer/scheduler state.")
    if payload.get("artifact_version")=="finance-minigpt-checkpoint-v3" and payload.get("checkpoint_kind")=="final" and payload.get("optimizer_state") is None: raise ValueError("Resumable final checkpoint must contain optimizer state.")
    model=FinanceMiniGPT(len(payload["vocabulary"]["chars"]),payload["architecture_config"]); model.load_state_dict(payload["model_state"])
    return model,payload

def compare_checkpoint_logits(reference,reconstructed,*,rtol=1e-6,atol=1e-7):
    if not isinstance(reference,torch.Tensor) or not isinstance(reconstructed,torch.Tensor): raise TypeError("Checkpoint logit comparison requires tensors.")
    comparison={"reference_device_before_normalization":str(reference.device),"reconstructed_device_before_normalization":str(reconstructed.device),"reference_shape":list(reference.shape),"reconstructed_shape":list(reconstructed.shape),"reference_dtype":str(reference.dtype),"reconstructed_dtype":str(reconstructed.dtype),"normalized_device":"cpu","rtol":rtol,"atol":atol}
    if reference.shape!=reconstructed.shape: raise AssertionError(f"Checkpoint logit shape mismatch: {comparison}")
    if reference.dtype!=reconstructed.dtype: raise AssertionError(f"Checkpoint logit dtype mismatch: {comparison}")
    reference_cpu=reference.detach().cpu(); reconstructed_cpu=reconstructed.detach().cpu()
    if not torch.isfinite(reference_cpu).all() or not torch.isfinite(reconstructed_cpu).all(): raise FloatingPointError("Checkpoint logits must be finite.")
    comparison["max_abs_logit_difference"]=float((reference_cpu-reconstructed_cpu).abs().max().detach().cpu())
    if torch.equal(reference_cpu,reconstructed_cpu): comparison["policy"]="exact"
    else:
        torch.testing.assert_close(reference_cpu,reconstructed_cpu,rtol=rtol,atol=atol); comparison["policy"]="torch.testing.assert_close"
    return comparison


In [ ]:
@torch.no_grad()
def generate(model,prompt,stoi,itos,max_new_tokens,temperature=0.0,top_k=None,seed=None,device=None):
    unknown=sorted(set(prompt)-set(stoi))
    if unknown: raise ValueError(f"Prompt contains unknown characters: {unknown!r}")
    if temperature<0: raise ValueError("temperature must be nonnegative.")
    if top_k is not None and top_k<1: raise ValueError("top_k must be positive.")
    device=device or next(model.parameters()).device; ids=torch.tensor([[stoi[ch] for ch in prompt]],dtype=torch.long,device=device)
    prior=model.training; model.eval(); gen=None
    if temperature>0: gen=torch.Generator(device=device.type).manual_seed(CONFIG["seed"] if seed is None else seed)
    try:
        for _ in range(max_new_tokens):
            context=ids[:,-model.architecture["block_size"]:]
            logits=model(context)["logits"][:,-1]
            if temperature==0: next_id=logits.argmax(-1,keepdim=True)
            else:
                logits=logits/temperature
                if top_k is not None:
                    threshold=torch.topk(logits,min(top_k,logits.size(-1))).values[:,-1,None]; logits=logits.masked_fill(logits<threshold,float("-inf"))
                next_id=torch.multinomial(F.softmax(logits,-1),1,generator=gen)
            ids=torch.cat((ids,next_id),dim=1)
    finally: model.train(prior)
    flat=ids[0].tolist(); return {"token_ids":flat,"text":"".join(itos[i] for i in flat)}

tiny_model.train(); greedy1=generate(tiny_model,EVAL_PROMPTS[0],stoi,itos,8,device=DEVICE)
train_mode_restored=tiny_model.training
greedy2=generate(tiny_model,EVAL_PROMPTS[0],stoi,itos,8,device=DEVICE)
stoch1=generate(tiny_model,EVAL_PROMPTS[0],stoi,itos,8,0.8,10,123,DEVICE)
stoch2=generate(tiny_model,EVAL_PROMPTS[0],stoi,itos,8,0.8,10,123,DEVICE)
tiny_model.eval(); generate(tiny_model,EVAL_PROMPTS[0],stoi,itos,1,device=DEVICE)
record_check("generation mode restoration", train_mode_restored and not tiny_model.training)
record_check("deterministic greedy generation", greedy1==greedy2)
record_check("seeded stochastic generation", stoch1==stoch2)
print("Generation mode restoration and reproducibility PASS (samples intentionally not displayed)")



In [ ]:
def character_kl(sample,reference,alphabet,eps=1e-12):
    """Lower is closer. Empty sample -> 0; empty reference with nonempty sample -> inf. Character frequency ignores nothing; smoothing limits zeros. Not semantic."""
    if not sample:return 0.0
    if not reference:return float("inf")
    p=np.array([sample.count(c) for c in alphabet],float); q=np.array([reference.count(c) for c in alphabet],float); p=(p+eps)/(p.sum()+eps*len(p)); q=(q+eps)/(q.sum()+eps*len(q)); return float(np.sum(p*np.log(p/q)))
def in_corpus_word_fraction(sample,reference_words):
    """Higher is more lexically familiar. No sample words -> 0. Case-sensitive whitespace words; not grammaticality."""
    words=sample.split(); return 0.0 if not words else sum(w in reference_words for w in words)/len(words)
def distinct_n(text,n):
    """Higher means more unique word n-grams. Fewer than n words -> 0. Sensitive to length/tokenization; not quality."""
    words=text.split(); grams=list(zip(*(words[i:] for i in range(n)))); return 0.0 if not grams else len(set(grams))/len(grams)
def repeated_word_trigram_rate(text):
    """Lower means fewer repeated word trigrams. No trigrams -> 0. Exact-match repetition only; may penalize legitimate phrases."""
    words=text.split(); grams=list(zip(words,words[1:],words[2:])); return 0.0 if not grams else (len(grams)-len(set(grams)))/len(grams)
def numerical_format_validity(text):
    """Higher means regex-detected numeric tokens have allowed finance-like forms. No numeric tokens -> 1. Regex coverage is limited, not factual validation."""
    candidates=re.findall(r"\S*\d\S*",text)
    if not candidates:return 1.0
    valid=re.compile(r"^[\$+-]?(?:\d+(?:\.\d+)?|\d+-\d+/\d+|\d+/\d+)(?:%|percent|basis)?[.,;:]?$")
    return sum(bool(valid.fullmatch(x)) for x in candidates)/len(candidates)

reference="Committee inflation 2 percent Committee policy"; score_sample="Committee inflation 2 percent"
scorecard_checks = (
    character_kl("",reference,chars)==0
    and in_corpus_word_fraction("",set(reference.split()))==0
    and distinct_n("a b a",1)==2/3
    and distinct_n("a b",2)==1
    and repeated_word_trigram_rate("a b c a b c")==1/4
    and numerical_format_validity("rate 2 percent and 1/4") == 1.0
)
record_check("scorecard unit checks", scorecard_checks)
print("Scorecard unit checks PASS")

def plot_attention(model,text,stoi,itos,layer,head,device=None):
    import matplotlib.pyplot as plt
    ids=encode(text); device=device or next(model.parameters()).device
    if len(ids)>model.architecture["block_size"]: raise ValueError("Visualization text exceeds block_size.")
    prior=model.training; model.eval()
    try:
        with torch.no_grad(): maps=model(torch.tensor([ids],dtype=torch.long,device=device),return_attentions=True)["attentions"]
    finally: model.train(prior)
    if not 0<=layer<len(maps): raise IndexError("layer is out of range.")
    if not 0<=head<maps[layer].size(1): raise IndexError("head is out of range.")
    matrix=maps[layer][0,head].detach().cpu().numpy(); labels=[f"{i}:{repr(itos[t])}" for i,t in enumerate(ids)]
    fig,ax=plt.subplots(figsize=(7,6)); image=ax.imshow(matrix,vmin=0,vmax=matrix.max(),cmap="viridis")
    ax.set(xticks=range(len(labels)),yticks=range(len(labels)),xticklabels=labels,yticklabels=labels,title=f"Causal attention — layer {layer}, head {head}",xlabel="key",ylabel="query")
    ax.tick_params(axis="x",rotation=90); fig.colorbar(image,ax=ax,label="attention probability"); fig.tight_layout(); return fig,ax


In [ ]:
# Phase 3A bounded optimizer/schedule/guard checks remain active.
optimizer_test_model=FinanceMiniGPT(VOCAB_SIZE,TINY_ARCH); test_weight_decay=0.1
test_optimizer,membership=configure_adamw(optimizer_test_model,1e-3,test_weight_decay); named=dict(optimizer_test_model.named_parameters())
decay_ids={id(p) for p in test_optimizer.param_groups[0]["params"]}; no_decay_ids={id(p) for p in test_optimizer.param_groups[1]["params"]}; all_ids={id(p) for p in optimizer_test_model.parameters() if p.requires_grad}
record_check("AdamW groups disjoint",decay_ids.isdisjoint(no_decay_ids)); record_check("AdamW full parameter coverage",decay_ids|no_decay_ids==all_ids)
record_check("AdamW group weight decay values",test_optimizer.param_groups[0]["weight_decay"]==test_weight_decay and test_optimizer.param_groups[1]["weight_decay"]==0.0)
record_check("AdamW LayerNorm and bias no-decay",all(group=="no_decay" for name,group in membership.items() if "ln" in name or name.endswith("bias")))
record_check("AdamW embedding and tied head no-decay",membership["token_embedding.weight"]=="no_decay" and optimizer_test_model.lm_head.weight.data_ptr()==optimizer_test_model.token_embedding.weight.data_ptr() and id(optimizer_test_model.token_embedding.weight) in no_decay_ids)
record_check("AdamW matrix weights decay",all(group=="decay" for name,group in membership.items() if named[name].ndim>=2 and name!="token_embedding.weight"))
for warmup in (0,3):
    values=[cosine_lr(step,10,warmup,1e-3,1e-4) for step in range(10)]
    record_check(f"cosine schedule finite nonnegative warmup={warmup}",all(math.isfinite(v) and v>=0 for v in values)); expected_start=1e-3 if warmup==0 else 1e-3/warmup
    record_check(f"cosine schedule start warmup={warmup}",math.isclose(values[0],expected_start,rel_tol=0,abs_tol=1e-15))
    if warmup:
        record_check("cosine warmup boundary",math.isclose(values[warmup-1],1e-3,rel_tol=0,abs_tol=1e-15)); record_check("cosine decay boundary",math.isclose(values[warmup],1e-3,rel_tol=0,abs_tol=1e-15))
    record_check(f"cosine schedule final warmup={warmup}",math.isclose(values[-1],1e-4,rel_tol=0,abs_tol=1e-15))
class GuardProbeDataset:
    sampled=False
    def sample_batch(self,*args,**kwargs): self.sampled=True; raise AssertionError("batch sampling occurred behind disabled guard")
guard_model=FinanceMiniGPT(VOCAB_SIZE,TINY_ARCH); guard_model.eval(); guard_ds=GuardProbeDataset(); guard_path=Path("/tmp/phase3a_guard_must_not_exist.pt")
guard_training={"lr":1e-3,"weight_decay":.1,"betas":[.9,.95],"seed":0,"steps":1,"warmup_steps":0,"min_lr":1e-4,"batch_size":1,"grad_clip":1.,"validation_steps":[0],"eval_batch_size":1,"best_path":guard_path,"final_path":guard_path}
guard_raised=False
try: train_model(guard_model,guard_ds,guard_ds,guard_training,{},torch.device("cpu"))
except RuntimeError as exc: guard_raised="authorized resolved run configuration" in str(exc)
record_check("canonical guard raises",guard_raised); record_check("canonical guard precedes side effects",not guard_model.training and not guard_ds.sampled and not guard_path.exists())
enter_training_mode(guard_model,True); record_check("training mode entered only after guard passes",guard_model.training); guard_model.eval()
print("AdamW grouping, cosine schedule, and canonical guard tests PASS")


In [ ]:
checkpoint_context={"vocabulary":VOCABULARY_DATA,"corpus_id":manifest["corpus_id"],"corpus_sha256":manifest["corpus_sha256"],"split_metadata":summary_rows,"seed":CONFIG["seed"]}
tmp_checkpoint=Path("/tmp/finance_minigpt_phase3a_tiny_checkpoint.pt")
tiny_model.eval(); save_checkpoint(tmp_checkpoint,"phase3a-test",tiny_model,None,None,{"purpose":"bounded reload test"},checkpoint_context,0,0,float(result["loss"].detach().cpu()),[])
reloaded,payload=load_checkpoint(tmp_checkpoint,map_location=DEVICE); reloaded.to(DEVICE).eval()
with torch.no_grad(): original_logits=tiny_model(sample)["logits"]; reloaded_logits=reloaded(sample)["logits"]
phase3a_reload_comparison=compare_checkpoint_logits(original_logits,reloaded_logits)
record_check("checkpoint reload equivalence", phase3a_reload_comparison["max_abs_logit_difference"]==0.0)
record_check("checkpoint corpus and vocabulary", payload["corpus_sha256"]==EXPECTED["sha256"] and payload["vocabulary"]["chars"]==chars)
record_check("checkpoint serialized parameter count", payload["architecture_config"]["parameter_count"] == sum(p.numel() for p in reloaded.parameters()))

expected_checks = {
    "body-only vocabulary exactness", "structural marker/header exclusion", "training vocabulary size",
    "tokenizer train round-trip", "validation character coverage", "prompt character coverage",
    "numerical attention shapes", "independent numerical attention reference", "attention rows normalize", "future attention is zero",
    "multi-head output shape", "block shape invariance", "logits shape", "finite scalar loss", "finite nonzero gradients",
    "optimizer step changes parameters", "causal non-leakage", "context overflow rejection", "weight tying storage",
    "serialized architecture parameter count", "GPT-style initialization statistics", "zero linear biases",
    "dropout train stochasticity", "dropout eval determinism", "AdamW groups disjoint", "AdamW full parameter coverage",
    "AdamW group weight decay values", "AdamW LayerNorm and bias no-decay", "AdamW embedding and tied head no-decay",
    "AdamW matrix weights decay", "cosine schedule finite nonnegative warmup=0", "cosine schedule start warmup=0",
    "cosine schedule final warmup=0", "cosine schedule finite nonnegative warmup=3", "cosine schedule start warmup=3",
    "cosine warmup boundary", "cosine decay boundary", "cosine schedule final warmup=3", "canonical guard raises",
    "canonical guard precedes side effects", "training mode entered only after guard passes", "generation mode restoration",
    "deterministic greedy generation", "seeded stochastic generation", "scorecard unit checks",
    "checkpoint reload equivalence", "checkpoint corpus and vocabulary", "checkpoint serialized parameter count",
}
missing=expected_checks-TEST_RESULTS.keys(); unexpected=TEST_RESULTS.keys()-expected_checks
if missing or unexpected: raise AssertionError(f"Phase 3A summary mismatch; missing={sorted(missing)}, unexpected={sorted(unexpected)}")
failed={name:passed for name,passed in TEST_RESULTS.items() if not passed}
if failed: raise AssertionError(f"Phase 3A failures: {failed}")
print("Phase 3A correctness suite PASS:",len(TEST_RESULTS),"derived checks")
print("Temporary checkpoint reload equivalence PASS:",tmp_checkpoint)


In [ ]:
SMOKE_FILENAMES={
    "best_checkpoint":"finance_minigpt_smoke_best.pt","final_checkpoint":"finance_minigpt_smoke_final.pt",
    "metrics":"finance_minigpt_smoke_metrics.csv","run_record":"finance_minigpt_smoke_run.json",
    "samples":"finance_minigpt_smoke_samples.json","training_plot":"finance_minigpt_smoke_training.png",
    "attention_plot":"finance_minigpt_smoke_attention.png",
}
CANONICAL_FILENAMES=dict(CONFIG["artifacts"])
def git_metadata(repo_root):
    branch=subprocess.run(["git","branch","--show-current"],cwd=repo_root,capture_output=True,text=True,check=True).stdout.strip()
    commit=subprocess.run(["git","rev-parse","HEAD"],cwd=repo_root,capture_output=True,text=True,check=True).stdout.strip()
    status=subprocess.run(["git","status","--porcelain"],cwd=repo_root,capture_output=True,text=True,check=True).stdout
    return {"branch":branch,"commit":commit,"clean":status=="","porcelain":status}

def validate_canonical_request(env,device_type,repo_root,*,git_clean=None,output_exists=None):
    errors=[]; raw_output=env.get("WEEK03_CANONICAL_OUTPUT_DIR")
    if env.get("WEEK03_RUN_MODE")!="phase4-canonical": errors.append("run mode")
    if env.get("WEEK03_CONFIRM_CANONICAL")!="YES": errors.append("confirmation")
    if device_type!="cuda": errors.append("CUDA device")
    if not raw_output: errors.append("explicit output variable")
    output=Path(raw_output).resolve() if raw_output else None; content_root=Path("/content").resolve(); repo_root=Path(repo_root).resolve()
    if output is not None:
        if output==content_root or not output.is_relative_to(content_root): errors.append("output beneath /content")
        if output==repo_root or output.is_relative_to(repo_root): errors.append("output outside repository")
        exists=output.exists() if output_exists is None else output_exists
        if exists: errors.append("absent output directory")
    clean=git_metadata(repo_root)["clean"] if git_clean is None else git_clean
    if not clean: errors.append("clean Git working tree")
    if errors: raise RuntimeError("Canonical authorization rejected: "+", ".join(errors))
    return output

def canonical_guard_self_tests():
    base={"WEEK03_RUN_MODE":"phase4-canonical","WEEK03_CONFIRM_CANONICAL":"YES","WEEK03_CANONICAL_OUTPUT_DIR":"/content/candidate"}; fake_repo=Path("/content/the-ai-engineer")
    cases={}
    def rejected(name,env=base,device="cuda",repo=fake_repo,clean=True,exists=False,needle=None):
        try: validate_canonical_request(env,device,repo,git_clean=clean,output_exists=exists)
        except RuntimeError as exc: cases[name]=needle in str(exc)
        else: cases[name]=False
    rejected("missing confirmation",{k:v for k,v in base.items() if k!="WEEK03_CONFIRM_CANONICAL"},needle="confirmation")
    rejected("non-CUDA device",device="cpu",needle="CUDA device")
    rejected("missing output variable",{k:v for k,v in base.items() if k!="WEEK03_CANONICAL_OUTPUT_DIR"},needle="explicit output variable")
    rejected("output outside /content",{**base,"WEEK03_CANONICAL_OUTPUT_DIR":"/tmp/candidate"},needle="output beneath /content")
    rejected("output inside repository",{**base,"WEEK03_CANONICAL_OUTPUT_DIR":"/content/the-ai-engineer/results"},needle="output outside repository")
    rejected("preexisting output directory",exists=True,needle="absent output directory")
    rejected("dirty repository",clean=False,needle="clean Git working tree")
    if not all(cases.values()): raise AssertionError(f"Canonical guard self-tests failed: {cases}")
    return cases

CANONICAL_GUARD_TESTS=canonical_guard_self_tests()
print("Canonical authorization guard tests PASS:",CANONICAL_GUARD_TESTS)

def resolve_smoke_output_dir():
    output=Path(os.environ.get("WEEK03_SMOKE_OUTPUT_DIR","/tmp/week03_finance_minigpt_smoke")).resolve()
    tmp_root=Path("/tmp").resolve(); repo_root=CORPUS_DIR.parents[1].resolve()
    if output==tmp_root or not output.is_relative_to(tmp_root): raise ValueError("Smoke output must be a dedicated directory beneath /tmp.")
    if output==repo_root or output.is_relative_to(repo_root): raise ValueError("Smoke output may not be inside the repository.")
    return output

def resolve_run_configuration(run_mode):
    repo_root=CORPUS_DIR.parents[1].resolve(); repo_before=git_metadata(repo_root)
    if run_mode=="phase3b-smoke":
        output=resolve_smoke_output_dir(); filenames=SMOKE_FILENAMES; architecture={"block_size":64,"d_model":128,"num_heads":4,"num_layers":2,"d_ff":512,"dropout":0.1}; training={"batch_size":8,"steps":20,"lr":5e-4,"min_lr":5e-5,"warmup_steps":2,"weight_decay":0.1,"betas":[0.9,0.95],"grad_clip":1.0,"seed":0,"validation_steps":[0,10,19],"eval_batch_size":128 if DEVICE.type in {"cuda","mps"} else 64}; generation_length=64; status="diagnostic-only"
    elif run_mode=="phase4-canonical":
        output=validate_canonical_request(os.environ,DEVICE.type,repo_root); filenames=CANONICAL_FILENAMES; architecture=dict(CANONICAL_ARCH); training=dict(CANONICAL_TRAINING); generation_length=200; status="canonical-candidate"
    else: raise ValueError("No training configuration for validation-only mode.")
    run_id=f"{run_mode}-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}-{uuid.uuid4().hex[:8]}"; paths={key:output/name for key,name in filenames.items()}
    resolved={"run_mode":run_mode,"run_id":run_id,"authorized_training":True,"status":status,"precision":"FP32","amp":False,"attention_implementation":"from-scratch","architecture":architecture,"batch_size":training["batch_size"],"steps":training["steps"],"lr":training["lr"],"min_lr":training["min_lr"],"warmup_steps":training["warmup_steps"],"weight_decay":training["weight_decay"],"betas":training["betas"],"grad_clip":training["grad_clip"],"seed":training["seed"],"validation_steps":training["validation_steps"],"eval_batch_size":training["eval_batch_size"],"tokens_per_step":training["batch_size"]*architecture["block_size"],"token_budget":training["steps"]*training["batch_size"]*architecture["block_size"],"generation":{"prompts":list(EVAL_PROMPTS),"continuation_characters":generation_length,"temperature":0.8,"top_k":20,"stochastic_seeds":[1000+i for i in range(len(EVAL_PROMPTS))]},"filenames":dict(filenames),"output_dir":str(output),"best_path":str(paths["best_checkpoint"]),"final_path":str(paths["final_checkpoint"])}
    if run_mode=="phase4-canonical": assert resolved["token_budget"]==CANONICAL_TOKEN_BUDGET
    return resolved,paths,repo_before


In [ ]:
if RUN_MODE=="validation-only":
    assert CONFIG["allow_canonical_training"] is False
    print("validation-only mode: training disabled; no canonical optimizer is constructed or stepped; frozen artifacts will be verified and rendered below.")
else:
    # Authorization and path validation complete before run-model construction or artifact creation.
    resolved_config,paths,repo_before=resolve_run_configuration(RUN_MODE); output_dir=Path(resolved_config["output_dir"]); run_id=resolved_config["run_id"]
    import matplotlib.pyplot as plt
    run_started=time.perf_counter(); output_dir.mkdir(parents=True,exist_ok=False)
    architecture=resolved_config["architecture"]; eval_batch_size=resolved_config["eval_batch_size"]
    seed_everything(resolved_config["seed"]); run_model=FinanceMiniGPT(VOCAB_SIZE,architecture).to(DEVICE)
    run_train_ds=DocumentWindowDataset(train_docs,architecture["block_size"],stoi); run_val_ds=DocumentWindowDataset(val_docs,architecture["block_size"],stoi)
    checkpoint_context={"vocabulary":VOCABULARY_DATA,"corpus_id":manifest["corpus_id"],"corpus_sha256":manifest["corpus_sha256"],"split_metadata":summary_rows,"seed":resolved_config["seed"],"verification_ids":encode(EVAL_PROMPTS[0])}
    training_result=train_model(run_model,run_train_ds,run_val_ds,resolved_config,checkpoint_context,DEVICE); metrics=training_result["metrics"]

    # Metrics CSV is the authoritative tabular source; blank validation fields mean no validation at that step.
    metric_fields=["step","train_minibatch_ce","gradient_norm_before_clipping","validation_ce","validation_perplexity","lr","tokens_processed","elapsed_seconds","throughput_tokens_per_second"]
    metrics_temp=paths["metrics"].with_name(paths["metrics"].name+".tmp")
    with metrics_temp.open("w",newline="",encoding="utf-8") as handle:
        writer=csv.DictWriter(handle,fieldnames=metric_fields); writer.writeheader(); writer.writerows(metrics)
    os.replace(metrics_temp,paths["metrics"])

    best_model,best_payload=load_checkpoint(paths["best_checkpoint"],map_location=DEVICE); best_model.to(DEVICE).eval()
    verification_tensor=torch.tensor([best_payload["verification_ids"]],dtype=torch.long,device=DEVICE)
    with torch.no_grad(): reloaded_verification=best_model(verification_tensor)["logits"]
    reload_comparison=compare_checkpoint_logits(best_payload["verification_logits"],reloaded_verification,rtol=1e-6,atol=1e-7)
    reload_max_abs_diff=reload_comparison["max_abs_logit_difference"]
    exhaustive=evaluate_exhaustive_details(best_model,run_val_ds,eval_batch_size,DEVICE)
    reconciliation_tolerance=1e-6
    reconciliation_difference=abs(exhaustive["cross_entropy"]-float(best_payload["best_validation_loss"])); assert reconciliation_difference<=reconciliation_tolerance

    generation_length=resolved_config["generation"]["continuation_characters"]; sample_records=[]; reference_text="\n".join(training_bodies); reference_words=set(reference_text.split())
    for prompt_index,prompt in enumerate(EVAL_PROMPTS):
        for method in ("greedy","temperature_top_k"):
            seed=None if method=="greedy" else 1000+prompt_index
            temperature=0.0 if method=="greedy" else 0.8; top_k=None if method=="greedy" else 20
            generated=generate(best_model,prompt,stoi,itos,generation_length,temperature,top_k,seed,DEVICE)
            prompt_ids=encode(prompt); continuation_ids=generated["token_ids"][len(prompt_ids):]; continuation=decode(continuation_ids)
            scores={"character_kl":character_kl(continuation,reference_text,chars),"in_corpus_word_fraction":in_corpus_word_fraction(continuation,reference_words),"distinct_1":distinct_n(continuation,1),"distinct_2":distinct_n(continuation,2),"repeated_word_trigram_rate":repeated_word_trigram_rate(continuation),"numerical_format_validity":numerical_format_validity(continuation)}
            sample_records.append({"prompt":prompt,"continuation":continuation,"full_text":generated["text"],"token_ids":generated["token_ids"],"continuation_token_ids":continuation_ids,"decoding":{"method":method,"max_new_tokens":generation_length,"temperature":temperature,"top_k":top_k},"checkpoint":{"filename":paths["best_checkpoint"].name,"kind":"best","run_id":run_id},"seed":seed,"scorecard":scores})
    reproducibility_note="Greedy decoding and validation should reproduce after reload. Seeded stochastic decoding should reproduce on the same PyTorch/device backend; exact samples are not guaranteed across CUDA, MPS, and CPU because backend sampling may differ. This does not imply model or checkpoint inconsistency."
    samples_payload={"schema_version":"finance-minigpt-samples-v2","run_mode":RUN_MODE,"run_id":run_id,"corpus_id":manifest["corpus_id"],"corpus_sha256":manifest["corpus_sha256"],"checkpoint":paths["best_checkpoint"].name,"validation":{"cross_entropy":exhaustive["cross_entropy"],"perplexity":exhaustive["perplexity"]},"reproducibility_limitation":reproducibility_note,"metric_notes":{"character_kl":"lower is closer in character distribution; not semantics","in_corpus_word_fraction":"higher is more lexically familiar; not grammaticality","distinct_1_2":"higher indicates lexical diversity; length-sensitive","repeated_word_trigram_rate":"lower indicates less exact repetition; may penalize valid repetition","numerical_format_validity":"higher indicates regex format validity; not factual validity"},"samples":sample_records}
    atomic_json_write(samples_payload,paths["samples"])

    valid_metrics=[row for row in metrics if row["validation_ce"] is not None]; best_row=min(valid_metrics,key=lambda row:row["validation_ce"])
    fig,(ax1,ax2)=plt.subplots(2,1,figsize=(8,7),sharex=True)
    ax1.plot([r["tokens_processed"] for r in metrics],[r["train_minibatch_ce"] for r in metrics],marker=".",label="training minibatch CE")
    ax1.plot([r["tokens_processed"] for r in valid_metrics],[r["validation_ce"] for r in valid_metrics],marker="o",label="exhaustive validation CE")
    ax1.scatter([best_row["tokens_processed"]],[best_row["validation_ce"]],marker="*",s=140,label="best checkpoint"); ax1.set_ylabel("cross-entropy"); ax1.legend(); ax1.grid(alpha=.25)
    ax2.plot([r["tokens_processed"] for r in metrics],[r["lr"] for r in metrics]); ax2.set(xlabel="tokens processed",ylabel="learning rate"); ax2.grid(alpha=.25); fig.suptitle(f"{RUN_MODE} training diagnostics"); fig.tight_layout()
    temp_plot=paths["training_plot"].with_name(paths["training_plot"].name+".tmp"); fig.savefig(temp_plot,dpi=120,format="png"); plt.close(fig); os.replace(temp_plot,paths["training_plot"])
    attention_text=val_docs[0]["body"][:32]; selected_layer=architecture["num_layers"]-1; selected_head=0; fig,_=plot_attention(best_model,attention_text,stoi,itos,layer=selected_layer,head=selected_head,device=DEVICE)
    fig.axes[0].set_title(f"{RUN_MODE} attention — last layer, head 0 (not causal explanation)"); temp_attention=paths["attention_plot"].with_name(paths["attention_plot"].name+".tmp"); fig.savefig(temp_attention,dpi=120,format="png"); plt.close(fig); os.replace(temp_attention,paths["attention_plot"])

    _,final_payload=load_checkpoint(paths["final_checkpoint"],map_location="cpu")
    other_keys=["best_checkpoint","final_checkpoint","metrics","samples","training_plot","attention_plot"]
    inventory={key:artifact_record(paths[key]) for key in other_keys}
    with paths["metrics"].open(newline="",encoding="utf-8") as handle: csv_rows=list(csv.DictReader(handle))
    csv_valid=[r for r in csv_rows if r["validation_ce"]]; csv_best=min(csv_valid,key=lambda r:float(r["validation_ce"]))
    png_ok=all(paths[key].read_bytes()[:8]==b"\x89PNG\r\n\x1a\n" for key in ("training_plot","attention_plot"))
    acceptance={
        "six_pre_run_json_artifacts_nonempty":all(paths[k].is_file() and paths[k].stat().st_size>0 for k in other_keys),
        "exact_filenames":{p.name for p in paths.values()}==set(resolved_config["filenames"].values()),
        "checkpoint_kinds":best_payload["checkpoint_kind"]=="best" and final_payload["checkpoint_kind"]=="final",
        "checkpoint_publication_policy":best_payload["optimizer_state"] is None and best_payload["scheduler_state"] is None and final_payload["optimizer_state"] is not None,
        "identity_agreement":best_payload["run_id"]==final_payload["run_id"]==run_id and best_payload["corpus_id"]==manifest["corpus_id"] and best_payload["corpus_sha256"]==manifest["corpus_sha256"] and best_payload["vocabulary"]==final_payload["vocabulary"]==VOCABULARY_DATA and best_payload["training_config"]==final_payload["training_config"]==resolved_config,
        "csv_best_agreement":int(csv_best["step"])==best_payload["best_step"] and abs(float(csv_best["validation_ce"])-best_payload["best_validation_loss"])<=reconciliation_tolerance,
        "reload_validation_agreement":reconciliation_difference<=reconciliation_tolerance,
        "samples_best_checkpoint_settings":all(s["checkpoint"]["filename"]==paths["best_checkpoint"].name and len(s["continuation_token_ids"])==generation_length for s in sample_records),
        "png_signatures":png_ok,
        "inventory_matches_files":all(inventory[k]==artifact_record(paths[k]) for k in other_keys),
        "mode_name_isolation":not any(("_smoke_" in p.name) for p in paths.values()) if RUN_MODE=="phase4-canonical" else not any((output_dir/name).exists() for name in CONFIG["artifacts"].values()),
        "candidate_outside_repository":not output_dir.resolve().is_relative_to(CORPUS_DIR.parents[1].resolve()),
    }
    failed=[name for name,passed in acceptance.items() if not passed]
    if failed: raise AssertionError(f"Smoke artifact acceptance failed: {failed}")
    repo_after=git_metadata(CORPUS_DIR.parents[1]); acceptance["repository_unmodified_during_run"]=repo_after==repo_before
    if not acceptance["repository_unmodified_during_run"]: raise AssertionError("Repository state changed during run.")
    elapsed=time.perf_counter()-run_started; throughput=training_result["tokens_processed"]/training_result["elapsed_seconds"]
    run_payload={"schema_version":"finance-minigpt-run-v2","run_status":resolved_config["status"],"run_mode":RUN_MODE,"run_id":run_id,"git":repo_before,"corpus":{"id":manifest["corpus_id"],"sha256":manifest["corpus_sha256"],"manifest_sha256":file_sha256(CORPUS_DIR/"fomc_statements_2015_2025_manifest.json"),"documents":len(documents),"train_documents":len(train_docs),"validation_documents":len(val_docs)},"architecture":best_payload["architecture_config"],"parameter_count":sum(p.numel() for p in best_model.parameters()),"resolved_configuration":resolved_config,"environment":environment_metadata(),"elapsed_seconds":elapsed,"training_elapsed_seconds":training_result["elapsed_seconds"],"steps_completed":resolved_config["steps"],"tokens_processed":training_result["tokens_processed"],"throughput_tokens_per_second":throughput,"best_step":training_result["best_step"],"final_step":resolved_config["steps"]-1,"best_validation_loss":training_result["best_validation_loss"],"exhaustive_validation":{**exhaustive,"reconciliation_difference":reconciliation_difference,"tolerance":reconciliation_tolerance},"reload":reload_comparison,"artifacts":inventory,"run_json_filename":paths["run_record"].name,"completion_status":"complete","acceptance_checks":acceptance,"limitations":["Distributional metrics do not establish coherence, factual correctness, prediction skill, or economic value.","Attention weight is not causal explanation."],"warnings":[reproducibility_note,"Canonical candidate artifacts require separate review before any Git promotion." if RUN_MODE=="phase4-canonical" else "Smoke diagnostics are not canonical results."]}
    atomic_json_write(run_payload,paths["run_record"])
    all_inventory={**inventory,"run_record":artifact_record(paths["run_record"])}
    assert all(paths[k].is_file() and paths[k].stat().st_size>0 for k in paths)
    written_run=json.loads(paths["run_record"].read_text(encoding="utf-8"))
    assert written_run["run_id"]==run_id and written_run["resolved_configuration"]==resolved_config
    assert written_run["best_step"]==int(csv_best["step"])==best_payload["best_step"] and abs(written_run["best_validation_loss"]-float(csv_best["validation_ce"]))<=reconciliation_tolerance
    assert all(written_run["artifacts"][k]==artifact_record(paths[k]) for k in other_keys)
    print(json.dumps({"run_id":run_id,"output_dir":str(output_dir),"device":str(DEVICE),"best_step":training_result["best_step"],"final_step":resolved_config["steps"]-1,"validation":exhaustive,"reload_max_abs_diff":reload_max_abs_diff,"samples":len(sample_records),"artifacts":all_inventory,"acceptance":acceptance},indent=2))


## 12. Canonical optimization trajectory

Everything below is loaded from the six frozen canonical artifacts; the current notebook execution does not optimize the model. Artifact hashes, schemas, run/corpus identities, metric rows, sample inventory, checkpoint metadata, PNG signatures, and all 13 acceptance checks must reconcile before any result is shown.

Cross-entropy and character perplexity measure next-character prediction, not sentence coherence or policy understanding. The best validation checkpoint occurs at the final recorded step, but selection is still governed by minimum validation CE rather than an assumption that the last step is always best.


In [ ]:
from html import escape

ADOPTED_RUN_ID = "phase4-canonical-20260720T141501Z-532748ca"
ADOPTED_GIT = {"branch": "capstone/week03-fomc", "commit": "b0c3903135779ec7ba78d43f89b859454458a4da"}
ADOPTED_MANIFEST_SHA256 = "3fc37f4a8ea0cc358269e7f8e37bf310cc96da4ba3981d361038a30a4511c2f2"
CANONICAL_ARTIFACTS = {
    "finance_minigpt_best.pt": (13_074_792, "c2b7d44b655d667382e7f318bf9f7eeabbdf25215adad22bd5449ff8292dfb49"),
    "finance_minigpt_metrics.csv": (135_567, "3f93d83aae4718f2e96a206601494814e54b028728cfb58d3a73c1272a03346e"),
    "finance_minigpt_samples.json": (64_372, "9196aa6e39284207dc7ea1d5aaf7ea8742235a6a6f0f19bca2b789437a42f37a"),
    "finance_minigpt_training.png": (89_714, "e2635e0d6bce985d16e2504a2d023fc9d2f7ac2fa5e4e92e84dc939e7a7e59c3"),
    "finance_minigpt_attention.png": (67_659, "ee6c9d8c2c2a3740bacf79e985c096f97056c73f5f0f1d5d3077d22a5f56230d"),
    "finance_minigpt_run.json": (6_368, "b8de4e910222b1eb8bc9a5e4af82d4d10aec8320b43194fa811e85a347b300ae"),
}

def resolve_canonical_dir() -> Path:
    candidate = CORPUS_DIR / "results" / "canonical"
    if not candidate.is_dir():
        raise FileNotFoundError(f"Canonical artifact directory not found: {candidate}")
    return candidate

CANONICAL_DIR = resolve_canonical_dir()
actual_names = {p.name for p in CANONICAL_DIR.iterdir()}
if actual_names != set(CANONICAL_ARTIFACTS):
    raise RuntimeError(f"Canonical artifact allowlist mismatch: {sorted(actual_names)}")
for name, (expected_size, expected_sha) in CANONICAL_ARTIFACTS.items():
    path = CANONICAL_DIR / name
    actual = (path.stat().st_size, file_sha256(path))
    if actual != (expected_size, expected_sha):
        raise RuntimeError(f"Canonical artifact integrity mismatch for {name}: {actual}")

canonical_run = json.loads((CANONICAL_DIR / "finance_minigpt_run.json").read_text(encoding="utf-8"))
canonical_samples = json.loads((CANONICAL_DIR / "finance_minigpt_samples.json").read_text(encoding="utf-8"))
with (CANONICAL_DIR / "finance_minigpt_metrics.csv").open(newline="", encoding="utf-8") as handle:
    canonical_metrics = list(csv.DictReader(handle))


In [ ]:
if canonical_run.get("schema_version") != "finance-minigpt-run-v2": raise RuntimeError("Unexpected run schema.")
if canonical_samples.get("schema_version") != "finance-minigpt-samples-v2": raise RuntimeError("Unexpected samples schema.")
if canonical_run.get("run_id") != ADOPTED_RUN_ID or canonical_samples.get("run_id") != ADOPTED_RUN_ID: raise RuntimeError("Canonical run ID mismatch.")
if canonical_run.get("run_mode") != "phase4-canonical" or canonical_run.get("run_status") != "canonical-candidate" or canonical_run.get("completion_status") != "complete": raise RuntimeError("Canonical mode/status/completion mismatch.")
if canonical_run.get("git", {}).get("branch") != ADOPTED_GIT["branch"] or canonical_run.get("git", {}).get("commit") != ADOPTED_GIT["commit"] or not canonical_run.get("git", {}).get("clean"): raise RuntimeError("Canonical Git provenance mismatch.")
if canonical_run.get("corpus", {}).get("id") != EXPECTED["corpus_id"] or canonical_run.get("corpus", {}).get("sha256") != EXPECTED["sha256"]: raise RuntimeError("Canonical corpus identity mismatch.")
if canonical_run.get("corpus", {}).get("manifest_sha256") != ADOPTED_MANIFEST_SHA256 or file_sha256(CORPUS_DIR / "fomc_statements_2015_2025_manifest.json") != ADOPTED_MANIFEST_SHA256: raise RuntimeError("Canonical manifest identity mismatch.")
expected_config = {"precision": "FP32", "amp": False, "attention_implementation": "from-scratch", "batch_size": 32, "steps": 1221, "lr": 3e-4, "min_lr": 3e-5, "warmup_steps": 61, "weight_decay": 0.1, "betas": [0.9, 0.95], "grad_clip": 1.0, "seed": 0, "validation_steps": [0, 250, 500, 750, 1000, 1220], "eval_batch_size": 128, "tokens_per_step": 4096, "token_budget": 5_001_216}
if any(canonical_run["resolved_configuration"].get(key) != value for key, value in expected_config.items()): raise RuntimeError("Canonical resolved configuration mismatch.")
if canonical_run.get("steps_completed") != 1221 or canonical_run.get("tokens_processed") != 5_001_216 or not canonical_run.get("best_step") == canonical_run.get("final_step") == 1220: raise RuntimeError("Canonical completion counters mismatch.")

recorded_files = {record["filename"]: (record["size_bytes"], record["sha256"]) for record in canonical_run["artifacts"].values()}
expected_pre_run = {name: values for name, values in CANONICAL_ARTIFACTS.items() if name != "finance_minigpt_run.json"}
expected_pre_run["finance_minigpt_final.pt"] = (38_546_611, "a6f2bc36986be45b17ce5013b00897606c9cb25956bf44af09154aaa02ddc82b")
if recorded_files != expected_pre_run: raise RuntimeError("run.json artifact inventory mismatch.")

metric_columns = ["step", "train_minibatch_ce", "gradient_norm_before_clipping", "validation_ce", "validation_perplexity", "lr", "tokens_processed", "elapsed_seconds", "throughput_tokens_per_second"]
if not canonical_metrics or list(canonical_metrics[0]) != metric_columns or len(canonical_metrics) != 1221: raise RuntimeError("Canonical metrics schema/row-count mismatch.")
metric_steps = [int(row["step"]) for row in canonical_metrics]
metric_tokens = [int(row["tokens_processed"]) for row in canonical_metrics]
if metric_steps != list(range(1221)) or any(b - a != 4096 for a, b in zip(metric_tokens, metric_tokens[1:])): raise RuntimeError("Canonical metric ordering mismatch.")
validation_rows = [row for row in canonical_metrics if row["validation_ce"]]
metric_best = min(validation_rows, key=lambda row: float(row["validation_ce"]))
if int(metric_best["step"]) != canonical_run["best_step"] or float(metric_best["validation_ce"]) != canonical_run["best_validation_loss"]: raise RuntimeError("Canonical best-step mismatch.")

if len(canonical_samples.get("samples", [])) != 10 or canonical_samples.get("checkpoint") != "finance_minigpt_best.pt": raise RuntimeError("Canonical sample inventory mismatch.")
if any(item.get("checkpoint", {}).get("run_id") != ADOPTED_RUN_ID for item in canonical_samples["samples"]): raise RuntimeError("Per-sample run identity mismatch.")
if any(len(item.get("continuation", "")) != 200 for item in canonical_samples["samples"]): raise RuntimeError("Canonical sample length mismatch.")
for index, item in enumerate(canonical_samples["samples"]):
    expected_method = "greedy" if index % 2 == 0 else "temperature_top_k"
    expected_seed = None if index % 2 == 0 else 1000 + index // 2
    expected_decoding = {"method": expected_method, "max_new_tokens": 200, "temperature": 0.0 if expected_method == "greedy" else 0.8, "top_k": None if expected_method == "greedy" else 20}
    if item.get("decoding") != expected_decoding or item.get("seed") != expected_seed: raise RuntimeError(f"Canonical decoding settings mismatch at sample {index + 1}.")
for png in ("finance_minigpt_training.png", "finance_minigpt_attention.png"):
    if (CANONICAL_DIR / png).read_bytes()[:8] != b"\x89PNG\r\n\x1a\n": raise RuntimeError(f"Invalid PNG signature: {png}")

from torch.serialization import safe_globals
from torch.torch_version import TorchVersion
with safe_globals([TorchVersion]):
    canonical_checkpoint = torch.load(CANONICAL_DIR / "finance_minigpt_best.pt", map_location="cpu", weights_only=True)
if canonical_checkpoint.get("artifact_version") != "finance-minigpt-checkpoint-v3" or canonical_checkpoint.get("checkpoint_kind") != "best": raise RuntimeError("Canonical checkpoint schema/kind mismatch.")
if canonical_checkpoint.get("run_id") != ADOPTED_RUN_ID or canonical_checkpoint.get("run_mode") != "phase4-canonical": raise RuntimeError("Canonical checkpoint identity mismatch.")
if canonical_checkpoint.get("optimizer_state") is not None or canonical_checkpoint.get("scheduler_state") is not None: raise RuntimeError("Promoted best checkpoint unexpectedly contains training state.")
if canonical_checkpoint.get("architecture_config") != canonical_run["architecture"] or canonical_checkpoint.get("parameter_count") != canonical_run["parameter_count"]: raise RuntimeError("Checkpoint architecture mismatch.")
if canonical_checkpoint.get("training_config") != canonical_run["resolved_configuration"]: raise RuntimeError("Checkpoint training configuration mismatch.")
if canonical_checkpoint.get("corpus_id") != EXPECTED["corpus_id"] or canonical_checkpoint.get("corpus_sha256") != EXPECTED["sha256"]: raise RuntimeError("Checkpoint corpus mismatch.")
if canonical_checkpoint.get("best_step") != canonical_run["best_step"] or canonical_checkpoint.get("best_validation_loss") != canonical_run["best_validation_loss"]: raise RuntimeError("Checkpoint result mismatch.")

acceptance = canonical_run.get("acceptance_checks", {})
if len(acceptance) != 13 or not all(type(value) is bool for value in acceptance.values()) or not all(acceptance.values()): raise RuntimeError("Canonical acceptance checks are not 13 true booleans.")
if canonical_run.get("reload", {}).get("max_abs_logit_difference") != 0.0: raise RuntimeError("Canonical reload result mismatch.")

print(f"Canonical artifact verification PASS: {ADOPTED_RUN_ID}")
print(f"Verified {len(CANONICAL_ARTIFACTS)} promoted artifacts; training was not invoked.")


In [ ]:
trajectory = [[
    int(row["step"]), f'{int(row["tokens_processed"]):,}', f'{float(row["validation_ce"]):.6f}', f'{float(row["validation_perplexity"]):.6f}', f'{float(row["lr"]):.8f}'
] for row in validation_rows]
display(Markdown(markdown_table(["Step", "Tokens", "Validation CE", "Character perplexity", "Learning rate"], trajectory)))
display(Image(filename=str(CANONICAL_DIR / "finance_minigpt_training.png")))

best_checkpoint_rows = [
    ("Run ID", canonical_run["run_id"]),
    ("Completion", canonical_run["completion_status"]),
    ("Best checkpoint", "finance_minigpt_best.pt"),
    ("Best step", canonical_run["best_step"]),
    ("Tokens processed", f'{canonical_run["tokens_processed"]:,}'),
    ("2025 validation CE", f'{canonical_run["exhaustive_validation"]["cross_entropy"]:.6f}'),
    ("Character perplexity", f'{canonical_run["exhaustive_validation"]["perplexity"]:.6f}'),
]
display(Markdown("### Best validation checkpoint\n\n" + markdown_table(["Field", "Canonical result"], best_checkpoint_rows)))


## 13. Checkpoint reload verification

The compact best checkpoint reconstructs the architecture and weights on a neutral device. Recorded probe logits and reloaded logits agree exactly after device normalization. This verifies serialization and reconstruction for that probe; it does not establish semantic competence.


In [ ]:
display(Markdown(
    f'**Reload equivalence:** maximum absolute logit difference = `{canonical_run["reload"]["max_abs_logit_difference"]}` '
    f'on shape `{canonical_run["reload"]["reference_shape"]}` after device normalization. '
    'This is an exact reconstruction check for the recorded probe, not evidence of semantic equivalence.'
))
acceptance_rows = [[name.replace("_", " "), "✅ true" if passed else "❌ false"] for name, passed in canonical_run["acceptance_checks"].items()]
display(Markdown("### Canonical acceptance checks (13/13)\n\n" + markdown_table(["Check", "Result"], acceptance_rows)))



## 14. Greedy and stochastic generations

All ten frozen generations are displayed without selection or rewriting. First come five greedy continuations, which repeatedly collapse into loops. Then come five seeded temperature/top-k continuations (`temperature=0.8`, `top_k=20`), which increase surface diversity but remain largely malformed. Decoding changes the sampling rule; it does not repair learned semantic limitations.


In [ ]:
samples_by_prompt = {}
for item in canonical_samples["samples"]:
    samples_by_prompt.setdefault(item["prompt"], {})[item["decoding"]["method"]] = item

for prompt, pair in samples_by_prompt.items():
    greedy = pair["greedy"]
    display(Markdown(
        f"#### {escape(prompt)} — greedy\n\n"
        f"<div style='white-space:pre-wrap; overflow-wrap:anywhere'><code>{escape(greedy['continuation'])}</code></div>"
    ))



In [ ]:
for prompt, pair in samples_by_prompt.items():
    stochastic = pair["temperature_top_k"]
    display(Markdown(
        f"#### {escape(prompt)} — temperature 0.8, top-k 20, seed {stochastic['seed']}\n\n"
        f"<div style='white-space:pre-wrap; overflow-wrap:anywhere'><code>{escape(stochastic['continuation'])}</code></div>"
    ))



## 15. Scorecard and semantic failure analysis

Character KL measures distributional distance; corpus-word fraction measures lexical overlap; distinct-n measures surface diversity; repeated-trigram rate flags one form of degeneration; numeric-format validity checks formatting only. None measures factuality, syntax, policy reasoning, or economic usefulness.

The scorecard confirms the visible trade-off: greedy decoding repeats more, while stochastic decoding is more diverse. Human inspection remains decisive—the passages are not reliably grammatical or semantically coherent. This negative semantic result is a model/data-scale limitation, not a software failure.


In [ ]:
score_rows = []
for index, item in enumerate(canonical_samples["samples"], 1):
    score = item["scorecard"]
    score_rows.append([
        index, item["prompt"], item["decoding"]["method"], item["seed"] if item["seed"] is not None else "—",
        f'{score["character_kl"]:.3f}', f'{score["in_corpus_word_fraction"]:.3f}',
        f'{score["distinct_1"]:.3f}', f'{score["distinct_2"]:.3f}',
        f'{score["repeated_word_trigram_rate"]:.3f}', f'{score["numerical_format_validity"]:.1f}',
    ])
display(Markdown(markdown_table(
    ["#", "Prompt", "Mode", "Seed", "Char KL ↓", "Corpus-word fraction ↑", "Distinct-1 ↑", "Distinct-2 ↑", "Repeated trigram ↓", "Numeric format"],
    score_rows,
)))



## 16. Attention visualization and masking caveat

The frozen heatmap shows attention allocation for one head, one layer, and one validation prompt. The empty upper-right triangle verifies causal masking: no query assigns probability to a future key.

The heatmap does not explain the model’s reasoning, identify causal features, or establish economic interpretability. Other heads, layers, and prompts can allocate attention differently.


In [ ]:
display(Image(filename=str(CANONICAL_DIR / "finance_minigpt_attention.png")))


## 17. Findings, limitations, and future research

The frozen experiment supports four separate conclusions:

- **Transformer implementation:** successful.
- **Numerical optimization:** successful; validation CE fell from 4.3814 to 1.8917.
- **Reproducibility and artifact reconciliation:** successful; reload equivalence and 13/13 acceptance checks passed.
- **Coherent semantic FOMC generation:** unsuccessful.

The small corpus is sufficient to demonstrate Transformer mechanics, loss optimization, chronological validation, and checkpointing. It is insufficient for reliable policy-language generation from scratch. The 2025 period was used for checkpoint selection, overlapping windows are not independent, and the scorecard is not a semantic evaluation.

Future experiments could study a larger policy-language corpus, subword tokenization, transfer learning or fine-tuning, scaling-law behavior, and stronger semantic evaluation. Those are new studies—not extensions of this frozen release, and not reasons to tune repeatedly against the existing 2025 validation set.
